In [1]:
import os
import sys
import warnings

# Suppress harmless pandas / arrow / incubator warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Configure environment to use /opt/spark
os.environ["SPARK_HOME"] = "/opt/spark"
sys.path.insert(0, "/opt/spark/python/lib/py4j-0.10.9.9-src.zip")
sys.path.insert(0, "/opt/spark/python")

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, round, avg, isnan

In [2]:
spark = (
    SparkSession.builder
    .appName("Student_Dropout_Risk_Phase1")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.fs.defaultFS", "file:///")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)

print(f"✅ PySpark version: {spark.version} (configured to local filesystem)")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/02 11:33:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/02 11:33:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✅ PySpark version: 4.2.0 (configured to local filesystem)


In [3]:
import os

# Load dataset from src/data/dataset.csv
data_path = f"file://{os.path.abspath('../src/data/dataset.csv')}"
print(f"Reading from: {data_path}")

df_raw = spark.read.csv(data_path, header=True, inferSchema=True)

total_students = df_raw.count()
total_columns = len(df_raw.columns)
print(f"✅ Successfully loaded! Total Students: {total_students:,}")
print(f"Total Columns: {total_columns}")

print("\n--- Raw Target Distribution ---")
df_raw.groupBy("Target").count().orderBy("count", ascending=False).show()

df_raw.printSchema()

Reading from: file:///home/adi/Documents/projects/student-dropout-prediction/src/data/dataset.csv
✅ Successfully loaded! Total Students: 4,424
Total Columns: 35

--- Raw Target Distribution ---
+--------+-----+
|  Target|count|
+--------+-----+
|Graduate| 2209|
| Dropout| 1421|
|Enrolled|  794|
+--------+-----+

root
 |-- Marital status: integer (nullable = true)
 |-- Application mode: integer (nullable = true)
 |-- Application order: integer (nullable = true)
 |-- Course: integer (nullable = true)
 |-- Daytime/evening attendance: integer (nullable = true)
 |-- Previous qualification: integer (nullable = true)
 |-- Nacionality: integer (nullable = true)
 |-- Mother's qualification: integer (nullable = true)
 |-- Father's qualification: integer (nullable = true)
 |-- Mother's occupation: integer (nullable = true)
 |-- Father's occupation: integer (nullable = true)
 |-- Displaced: integer (nullable = true)
 |-- Educational special needs: integer (nullable = true)
 |-- Debtor: integer (nu

In [4]:
# Map Target to Binary Dropout Risk:
# 1.0 = Dropout (At Risk)
# 0.0 = Graduate or Enrolled (Retained / Safe)
df_clean = df_raw.withColumn(
    "label",
    when(col("Target") == "Dropout", 1.0).otherwise(0.0)
)

# Cast categorical codes to StringType for proper StringIndexer encoding
cat_cols_to_cast = [
    "Course",
    "Daytime/evening attendance",
    "Marital status",
    "Gender",
    "Scholarship holder",
    "Tuition fees up to date",
    "Debtor",
    "Displaced"
]

for c in cat_cols_to_cast:
    df_clean = df_clean.withColumn(c, col(c).cast("string"))

# Check for null/missing values across all columns
null_counts = [
    df_clean.filter(col(c).isNull() | isnan(c)).count() if t in ["double", "float"]
    else df_clean.filter(col(c).isNull()).count()
    for c, t in df_clean.dtypes
]

null_report = [(c, n) for (c, _), n in zip(df_clean.dtypes, null_counts) if n > 0]
if null_report:
    print("Columns with Null Values:", null_report)
else:
    print("✅ Zero null values detected! Dataset is clean and ready for modeling.")

df_clean.select("Course", "Daytime/evening attendance", "Target", "label").show(5)



✅ Zero null values detected! Dataset is clean and ready for modeling.
+------+--------------------------+--------+-----+
|Course|Daytime/evening attendance|  Target|label|
+------+--------------------------+--------+-----+
|     2|                         1| Dropout|  1.0|
|    11|                         1|Graduate|  0.0|
|     5|                         1| Dropout|  1.0|
|    15|                         1|Graduate|  0.0|
|     3|                         0|Graduate|  0.0|
+------+--------------------------+--------+-----+
only showing top 5 rows


In [5]:
print("--- Dropout Rate by Study Mode (1 = Daytime, 0 = Evening) ---")
df_clean.groupBy("Daytime/evening attendance").agg(
    count("*").alias("Total_Students"),
    round(avg("label") * 100, 2).alias("Dropout_Rate_Pct")
).orderBy("Dropout_Rate_Pct", ascending=False).show()

print("--- Dropout Rate by Scholarship Status (1 = Yes, 0 = No) ---")
df_clean.groupBy("Scholarship holder").agg(
    count("*").alias("Total_Students"),
    round(avg("label") * 100, 2).alias("Dropout_Rate_Pct")
).orderBy("Dropout_Rate_Pct", ascending=False).show()

print("--- Dropout Rate by Tuition Fees Up to Date (1 = Yes, 0 = No) ---")
df_clean.groupBy("Tuition fees up to date").agg(
    count("*").alias("Total_Students"),
    round(avg("label") * 100, 2).alias("Dropout_Rate_Pct")
).orderBy("Dropout_Rate_Pct", ascending=False).show()

print("--- Top 5 Academic Courses with Highest Dropout Rates ---")
df_clean.groupBy("Course").agg(
    count("*").alias("Total_Students"),
    round(avg("label") * 100, 2).alias("Dropout_Rate_Pct")
).orderBy("Dropout_Rate_Pct", ascending=False).show(5)


--- Dropout Rate by Study Mode (1 = Daytime, 0 = Evening) ---
+--------------------------+--------------+----------------+
|Daytime/evening attendance|Total_Students|Dropout_Rate_Pct|
+--------------------------+--------------+----------------+
|                         0|           483|           42.86|
|                         1|          3941|            30.8|
+--------------------------+--------------+----------------+

--- Dropout Rate by Scholarship Status (1 = Yes, 0 = No) ---
+------------------+--------------+----------------+
|Scholarship holder|Total_Students|Dropout_Rate_Pct|
+------------------+--------------+----------------+
|                 0|          3325|           38.71|
|                 1|          1099|           12.19|
+------------------+--------------+----------------+

--- Dropout Rate by Tuition Fees Up to Date (1 = Yes, 0 = No) ---
+-----------------------+--------------+----------------+
|Tuition fees up to date|Total_Students|Dropout_Rate_Pct|
+--------

In [6]:
import plotly.express as px
import plotly.io as pio

# Force Plotly to display inline inside Jupyter Notebook
pio.renderers.default = "iframe"

# 1. Dropout Rate by Tuition & Scholarship Status
summary_df = (
    df_clean.groupBy("Scholarship holder", "Tuition fees up to date")
    .agg(round(avg("label") * 100, 2).alias("Dropout_Rate_Pct"))
    .toPandas()
)

fig1 = px.bar(
    summary_df,
    x="Scholarship holder",
    y="Dropout_Rate_Pct",
    color="Tuition fees up to date",
    barmode="group",
    title="Dropout Rate (%) by Scholarship & Tuition Fee Status",
    labels={"Scholarship holder": "Has Scholarship (1=Yes, 0=No)", "Dropout_Rate_Pct": "Dropout Rate (%)"}
)
fig1.show()

# 2. Approved Units Distribution: Dropouts vs. Graduates/Enrolled
sample_eval = df_clean.select("label", "Curricular units 1st sem (approved)", "Curricular units 2nd sem (approved)").toPandas()
sample_eval["Status"] = sample_eval["label"].map({1.0: "Dropout (At Risk)", 0.0: "Retained"})

fig2 = px.box(
    sample_eval,
    x="Status",
    y="Curricular units 1st sem (approved)",
    color="Status",
    title="1st Semester Approved Units: Dropouts vs. Retained Students"
)
fig2.show()

In [7]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Categorical columns
categorical_cols = [
    "Course",
    "Daytime/evening attendance",
    "Marital status",
    "Gender",
    "Scholarship holder",
    "Tuition fees up to date",
    "Debtor",
    "Displaced"
]

# Numeric academic performance & engagement features
numeric_cols = [
    "Curricular units 1st sem (evaluations)",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 2nd sem (evaluations)",
    "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)",
    "Age at enrollment",
    "Unemployment rate",
    "Inflation rate",
    "GDP"
]

indexed_cols = [f"{c}_idx" for c in categorical_cols]
encoded_cols = [f"{c}_vec" for c in categorical_cols]

# 1. StringIndexer
indexer = StringIndexer(
    inputCols=categorical_cols,
    outputCols=indexed_cols,
    handleInvalid="keep"
)

# 2. OneHotEncoder
encoder = OneHotEncoder(
    inputCols=indexed_cols,
    outputCols=encoded_cols
)

# 3. VectorAssembler
assembler_inputs = numeric_cols + encoded_cols
assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features",
    handleInvalid="skip"
)

print(f"✅ Feature pipeline configured! Assembly inputs: {len(assembler_inputs)} features.")


✅ Feature pipeline configured! Assembly inputs: 18 features.


In [8]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

# 80% Train, 20% Holdout Test Split
train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)
print(f"Training samples: {train_df.count():,}, Holdout Testing samples: {test_df.count():,}")

# Model 1: Logistic Regression Pipeline
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=30,
    regParam=0.01,
    elasticNetParam=0.5
)
pipeline_lr = Pipeline(stages=[indexer, encoder, assembler, lr])

# Model 2: Random Forest Classifier Pipeline
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=80,
    maxDepth=8,
    seed=42
)
pipeline_rf = Pipeline(stages=[indexer, encoder, assembler, rf])

print("Training Logistic Regression Model...")
model_lr = pipeline_lr.fit(train_df)

print("Training Random Forest Classifier Model...")
model_rf = pipeline_rf.fit(train_df)
print("✅ Both classification models trained successfully!")


26/11/02 11:33:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Training samples: 3,581, Holdout Testing samples: 843
Training Logistic Regression Model...
Training Random Forest Classifier Model...


26/11/02 11:33:23 WARN DAGScheduler: Broadcasting large task binary with size 1337.8 KiB
26/11/02 11:33:23 WARN DAGScheduler: Broadcasting large task binary with size 1941.2 KiB


✅ Both classification models trained successfully!


In [9]:
import pandas as pd
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Score test set
preds_lr = model_lr.transform(test_df)
preds_rf = model_rf.transform(test_df)

# Initialize Spark evaluators
eval_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
eval_precision = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
eval_recall = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
eval_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Compile side-by-side benchmark comparison
benchmark_data = {
    "Evaluation Metric": [
        "ROC-AUC (Discrimination Power)",
        "F1-Score (Balanced Accuracy)",
        "Precision (Positive Predictive Value)",
        "Recall (Dropout Sensitivity)"
    ],
    "Logistic Regression": [
        eval_auc.evaluate(preds_lr),
        eval_f1.evaluate(preds_lr),
        eval_precision.evaluate(preds_lr),
        eval_recall.evaluate(preds_lr)
    ],
    "Random Forest Classifier": [
        eval_auc.evaluate(preds_rf),
        eval_f1.evaluate(preds_rf),
        eval_precision.evaluate(preds_rf),
        eval_recall.evaluate(preds_rf)
    ]
}

benchmark_table = pd.DataFrame(benchmark_data)
benchmark_table

26/11/02 11:33:25 WARN DAGScheduler: Broadcasting large task binary with size 1496.0 KiB
26/11/02 11:33:26 WARN DAGScheduler: Broadcasting large task binary with size 1508.0 KiB
26/11/02 11:33:26 WARN DAGScheduler: Broadcasting large task binary with size 1508.0 KiB
26/11/02 11:33:26 WARN DAGScheduler: Broadcasting large task binary with size 1508.0 KiB


,Evaluation Metric,Logistic Regression,Random Forest Classifier
0,ROC-AUC (Discrimination Power),0.893828,0.908177
1,F1-Score (Balanced Accuracy),0.853704,0.858385
2,Precision (Positive Predictive Value),0.863092,0.865018
3,Recall (Dropout Sensitivity),0.860024,0.863582


In [10]:
from pyspark.ml.functions import vector_to_array

# Extract continuous probability of dropping out (index 1 in the probability vector)
scored_test = preds_rf.withColumn("dropout_probability", vector_to_array(col("probability"))[1])

# Apply 3-Tier Early Warning Risk Categorization:
# High Risk:     P >= 0.60 (Immediate advisor intervention needed)
# Moderate Risk: 0.35 <= P < 0.60 (Monitor closely & tutoring support)
# Low Risk:      P < 0.35 (On track to graduate)
scored_test = scored_test.withColumn(
    "risk_level",
    when(col("dropout_probability") >= 0.60, "High Risk")
    .when(col("dropout_probability") >= 0.35, "Moderate Risk")
    .otherwise("Low Risk")
)

print("Sample Dropout Risk Predictions:")
scored_test.select(
    "Course",
    "Daytime/evening attendance",
    "Curricular units 1st sem (approved)",
    "Curricular units 2nd sem (approved)",
    "dropout_probability",
    "risk_level",
    "prediction",
    "label"
).show(10)



Sample Dropout Risk Predictions:
+------+--------------------------+-----------------------------------+-----------------------------------+--------------------+----------+----------+-----+
|Course|Daytime/evening attendance|Curricular units 1st sem (approved)|Curricular units 2nd sem (approved)| dropout_probability|risk_level|prediction|label|
+------+--------------------------+-----------------------------------+-----------------------------------+--------------------+----------+----------+-----+
|    10|                         1|                                  6|                                  6| 0.08309793867956561|  Low Risk|       0.0|  0.0|
|    10|                         1|                                  6|                                  5| 0.06547610392548035|  Low Risk|       0.0|  0.0|
|    10|                         1|                                  6|                                  6| 0.15823148381174543|  Low Risk|       0.0|  0.0|
|    10|                 

26/11/02 11:33:26 WARN DAGScheduler: Broadcasting large task binary with size 1507.2 KiB


In [11]:
export_cols = [
    "Course",
    "Daytime/evening attendance",
    "Scholarship holder",
    "Tuition fees up to date",
    "Debtor",
    "Age at enrollment",
    "Curricular units 1st sem (evaluations)",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 2nd sem (evaluations)",
    "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)",
    "dropout_probability",
    "risk_level",
    "prediction",
    "label"
]

os.makedirs("../output", exist_ok=True)
pdf_export = scored_test.select(export_cols).toPandas()
csv_output_path = "../output/test_dropout_predictions.csv"
pdf_export.to_csv(csv_output_path, index=False)
print(f"✅ Successfully exported {len(pdf_export):,} scored student predictions to: {csv_output_path}")


✅ Successfully exported 843 scored student predictions to: ../output/test_dropout_predictions.csv


26/11/02 11:33:27 WARN DAGScheduler: Broadcasting large task binary with size 1505.5 KiB


In [12]:

# 1. Risk Level Distribution
risk_counts = pdf_export["risk_level"].value_counts().reset_index()
risk_counts.columns = ["Risk Tier", "Number of Students"]

fig_pie = px.pie(
    risk_counts,
    names="Risk Tier",
    values="Number of Students",
    color="Risk Tier",
    color_discrete_map={"High Risk": "#d62728", "Moderate Risk": "#ff7f0e", "Low Risk": "#2ca02c"},
    title="Cohort Distribution by Dropout Risk Tier"
)
fig_pie.show()

# 2. Predicted Probability vs. Actual Status
pdf_export["Actual Status"] = pdf_export["label"].map({1.0: "Dropout", 0.0: "Retained"})
fig_hist = px.histogram(
    pdf_export,
    x="dropout_probability",
    color="Actual Status",
    nbins=40,
    title="Predicted Dropout Probability Distribution by Actual Status",
    labels={"dropout_probability": "Predicted Probability of Dropping Out"}
)
fig_hist.show()
